# Exploring the NYT Books API

## Importing packages and API key

In [1]:
# Importing important packages
from dotenv import load_dotenv
import os
import requests
import pandas as pd
from time import sleep
load_dotenv()

True

## Testing out the API call with the most recent bestsellers list

In [4]:
# This gets me the current list of fiction best sellers
url = f"https://api.nytimes.com/svc/books/v3/lists/current/combined-print-and-e-book-fiction.json?api-key={api_key}"
response = requests.get(url)

In [5]:
test = response.json()

In [6]:
test

{'status': 'OK',
 'copyright': 'Copyright (c) 2026 The New York Times Company. All Rights Reserved.',
 'num_results': 15,
 'last_modified': '2026-07-15T16:43:39Z',
 'results': {'display_name': 'Combined Print & E-Book Fiction',
  'list_name': 'Combined Print & E-Book Fiction',
  'list_name_encoded': 'combined-print-and-e-book-fiction',
  'previous_published_date': '2026-07-19',
  'published_date': '2026-07-26',
  'bestsellers_date': '2026-07-11',
  'normal_list_ends_at': 15,
  'updated': 'WEEKLY',
  'list_id': 704,
  'uri': 'nyt://bestsellerslist/de30e831-a20d-56d4-8f88-695162dc9183',
  'books': [{'age_group': '',
    'amazon_product_url': 'https://www.amazon.com/dp/1668236516?tag=thenewyorktim-20',
    'article_chapter_link': '',
    'asterisk': 0,
    'author': 'Allen Levi',
    'book_image': 'https://static01.nyt.com/bestsellers/images/9781668236512.jpg',
    'book_image_height': 0,
    'book_image_width': 0,
    'book_review_link': '',
    'book_uri': 'nyt://book/f3e1f4c8-34a3-54a3

The call returns 15 lists of books.

In [18]:
len(test["results"]["books"])

15

The description has one sentence that usually mentions the protagonist, so I plan to use this.

In [21]:
test["results"]["books"][0]["description"]

'A man travels to a small Southern town, where he purchases pencil drawings of local residents and exchanges them for stories.'

In [28]:
df = pd.json_normalize(test["results"]["books"])

In [33]:
df["description"]

0     A man travels to a small Southern town, where ...
1     Natalie Heller Mills, a privileged tradwife so...
2     Decades after their time together, a woman rec...
3     Ryland Grace awakes from a long sleep alone an...
4     The 25th book in the Scot Harvath series. Amer...
5     Hannah Wells makes a deal with the captain of ...
6     As things get tougher during the Depression, s...
7     A Coast Guard vet named Carl and his ex-girlfr...
8     After her husband ends their marriage and she ...
9     The second book in the Off-Campus series. A co...
10    The third book in the Off-Campus series. A bro...
11    Letters from someone she used to know push Syb...
12    A widow working the night shift at the Sowell ...
13    Sophie Drear is hired to revitalize a manor in...
14    After her fiancé leaves her, Frankie goes on h...
Name: description, dtype: str

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   age_group             15 non-null     str   
 1   amazon_product_url    15 non-null     str   
 2   article_chapter_link  15 non-null     str   
 3   asterisk              15 non-null     int64 
 4   author                15 non-null     str   
 5   book_image            15 non-null     str   
 6   book_image_height     15 non-null     int64 
 7   book_image_width      15 non-null     int64 
 8   book_review_link      15 non-null     str   
 9   book_uri              15 non-null     str   
 10  contributor           15 non-null     str   
 11  contributor_note      15 non-null     str   
 12  created_date          15 non-null     str   
 13  dagger                15 non-null     int64 
 14  description           15 non-null     str   
 15  first_chapter_link    15 non-null     str   
 16  pri

## Getting the data from the NYT API

In [29]:
# Setting a variable to make sure the loop doesn't use all my API calls 
number_of_calls = 300

# Setting up an empty url variable for my calls
url = ""

# Setting up an empty variable for the date of the previous list
previous_published_date = ""

for i in range(number_of_calls):
    
    # if it is the first call, use the specific request for the current list so that we can get the dates
    if i == 0:
        url = f"https://api.nytimes.com/svc/books/v3/lists/current/combined-print-and-e-book-fiction.json?api-key={api_key}"
        response = requests.get(url)
        bestseller_list = response.json()

        # getting the date info I need for the next call
        previous_published_date = bestseller_list["results"]["previous_published_date"]
        # print(previous_published_date)
        
        # Getting the dataset I need
        bestselling_books = bestseller_list["results"]["books"]
        # print(bestselling_books[0])

        # Create a dataframe with the list of books
        bestselling_books_df = pd.json_normalize(bestselling_books)

        # Adding the additional columns with dates that I need
        published_date = bestseller_list["results"]["published_date"]
        bestselling_books_df["published_date"] = published_date

        bestsellers_date = bestseller_list["results"]["bestsellers_date"]
        bestselling_books_df["bestsellers_date"] = bestsellers_date

        # Since this will run a while, I am going to print to let myself know it worked
        print("Call 1 done")

        # Adding 12 seconds of sleep so as to not exceed the NYT call limit
        sleep(12)
        
    # For the answer other than the first call, use the date previous_published_date to call the previous list
    else: 
        url = f"https://api.nytimes.com/svc/books/v3/lists/{previous_published_date}/combined-print-and-e-book-fiction.json?api-key={api_key}"
        response = requests.get(url)
        bestseller_list = response.json()

        # getting the date info I need
        previous_published_date = bestseller_list["results"]["previous_published_date"]
        # print(previous_published_date)

        # Printing the info I need to check this worked
        bestselling_books = bestseller_list["results"]["books"]
        # print(bestselling_books[0])

        # Creating a new dataframe with the new list 
        new_bestselling_books_df = pd.json_normalize(bestselling_books)

        published_date = bestseller_list["results"]["published_date"]
        new_bestselling_books_df["published_date"] = published_date

        bestsellers_date = bestseller_list["results"]["bestsellers_date"]
        new_bestselling_books_df["bestsellers_date"] = bestsellers_date

        # Appending the new dataframe to the master data frame
        bestselling_books_df = pd.concat([bestselling_books_df, new_bestselling_books_df])

        # Since this will run a while, I am going to print to let myself know it worked
        print(f"Call {i} done")

        # Adding 12 seconds of sleep so as to not exceed the NYT call limit
        sleep(12)
        

Call 1 done
Call 1 done
Call 2 done
Call 3 done
Call 4 done
Call 5 done
Call 6 done
Call 7 done
Call 8 done
Call 9 done
Call 10 done
Call 11 done
Call 12 done
Call 13 done
Call 14 done
Call 15 done
Call 16 done
Call 17 done
Call 18 done
Call 19 done
Call 20 done
Call 21 done
Call 22 done
Call 23 done
Call 24 done
Call 25 done
Call 26 done
Call 27 done
Call 28 done
Call 29 done
Call 30 done
Call 31 done
Call 32 done
Call 33 done
Call 34 done
Call 35 done
Call 36 done
Call 37 done
Call 38 done
Call 39 done
Call 40 done
Call 41 done
Call 42 done
Call 43 done
Call 44 done
Call 45 done
Call 46 done
Call 47 done
Call 48 done
Call 49 done
Call 50 done
Call 51 done
Call 52 done
Call 53 done
Call 54 done
Call 55 done
Call 56 done
Call 57 done
Call 58 done
Call 59 done
Call 60 done
Call 61 done
Call 62 done
Call 63 done
Call 64 done
Call 65 done
Call 66 done
Call 67 done
Call 68 done
Call 69 done
Call 70 done
Call 71 done
Call 72 done
Call 73 done
Call 74 done
Call 75 done
Call 76 done
Call 77 d

Checking that it worked

In [30]:
 bestselling_books_df.shape

(4500, 30)

In [34]:
bestselling_books_df.head()

,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,book_uri,...,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links,published_date,bestsellers_date
0,,https://www.amazon.com/dp/1668236516?tag=thene...,,0,Allen Levi,https://static01.nyt.com/bestsellers/images/97...,0,0,,nyt://book/f3e1f4c8-34a3-54a3-8305-277666e21b4b,...,1,1,,THEO OF GOLDEN,2026-03-04T23:40:20.88Z,30,"[{'isbn10': '', 'isbn13': '9781668236512'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2026-07-12,2026-06-27
1,,https://www.amazon.com/dp/059380421X?tag=thene...,,0,Caro Claire Burke,https://static01.nyt.com/bestsellers/images/97...,400,312,,nyt://book/fb5cbc76-606c-50f7-b449-e958c73d2681,...,2,2,,YESTERYEAR,2026-04-16T20:39:40.59Z,12,"[{'isbn10': '', 'isbn13': '9780593804216'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2026-07-12,2026-06-27
2,,https://www.amazon.com/dp/0063511630?tag=thene...,,0,Ann Patchett,https://static01.nyt.com/bestsellers/images/97...,400,312,,nyt://book/6480f0ac-57e1-5272-9566-118c913fe058,...,3,3,,WHISTLER,2026-06-10T21:51:30.058Z,4,"[{'isbn10': '', 'isbn13': '9780063511637'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2026-07-12,2026-06-27
3,,https://www.amazon.com/dp/1954118813?tag=thene...,,0,Kathryn Stockett,https://static01.nyt.com/bestsellers/images/97...,400,312,,nyt://book/37a1d1e0-131c-5003-8a26-827e5b166569,...,4,7,,THE CALAMITY CLUB,2026-05-13T21:30:50.865Z,8,"[{'isbn10': '', 'isbn13': '9781954118812'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2026-07-12,2026-06-27
4,,https://www.amazon.com/dp/0593135202?tag=thene...,,0,Andy Weir,https://static01.nyt.com/bestsellers/images/97...,400,312,,nyt://book/c11ceffc-2fff-5cb0-98c7-c4a75b2d16bd,...,5,4,,PROJECT HAIL MARY,2026-05-20T22:02:25.385Z,55,"[{'isbn10': '', 'isbn13': '9780593135228'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2026-07-12,2026-06-27


In [32]:
bestselling_books_df.tail()

,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,book_uri,...,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links,published_date,bestsellers_date
10,,https://www.amazon.com/dp/0062956388?tag=thene...,,0,Lynsay Sands,https://static01.nyt.com/bestsellers/images/97...,500,331,,nyt://book/a37aebd8-9bd3-50ad-a2fd-27f8882ee73e,...,11,0,,IMMORTAL ANGEL,2026-03-06T09:04:51.486Z,1,"[{'isbn10': '', 'isbn13': '9780062956279'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
11,,https://www.amazon.com/dp/1496729129?tag=thene...,,0,Joanne Fluke,https://static01.nyt.com/bestsellers/images/97...,500,354,,nyt://book/c628e428-03bc-5bc2-9e33-3827259145b6,...,12,0,,CHRISTMAS CUPCAKE MURDER,2026-03-06T09:04:52.682Z,1,"[{'isbn10': '', 'isbn13': '9781496729149'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
12,,https://www.amazon.com/dp/0316498939?tag=thene...,,0,Robert Galbraith,https://static01.nyt.com/bestsellers/images/97...,500,330,,nyt://book/d0520287-a2d9-5c36-8f21-71de1f9138f2,...,13,8,,TROUBLED BLOOD,2026-03-06T08:56:38.657Z,3,"[{'isbn10': '', 'isbn13': '9780316498968'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
13,,https://www.amazon.com/dp/150118881X?tag=thene...,,0,Ruth Ware,https://static01.nyt.com/bestsellers/images/97...,500,331,,nyt://book/603b3844-b7e7-5f0b-b32c-0d760002f659,...,14,10,,ONE BY ONE,2026-03-06T09:09:02.774Z,4,"[{'isbn10': '', 'isbn13': '9781501188817'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
14,,https://www.amazon.com/dp/0525559477?tag=thene...,,0,Matt Haig,https://static01.nyt.com/bestsellers/images/97...,500,331,,nyt://book/60d0ee2d-3d05-50c9-a484-050d17a2308e,...,15,0,,THE MIDNIGHT LIBRARY,2026-03-06T03:15:23.815Z,1,"[{'isbn10': '', 'isbn13': '9780525559474'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03


## Exporting the data to a csv

In [33]:
bestselling_books_df.to_csv("raw_data/raw_data.csv")

## Repeating the process across multiple days of data collection  

Gathering even more historical data the next day (API calls are limited to nyt)


In [44]:
# Setting a variable to make sure the loop doesn't use all my API calls 
number_of_calls = 450

# Setting up the last date I have
current = "2020-10-18"

# Setting up an empty url variable for my calls
url = ""

# Setting up an empty variable for the date of the previous list
previous_published_date = ""

for i in range(number_of_calls):
    
    # if it is the first call, use the specific request for the current list so that we can get the dates
    if i == 0:
        url = f"https://api.nytimes.com/svc/books/v3/lists/{current}/combined-print-and-e-book-fiction.json?api-key={api_key}"
        response = requests.get(url)
        bestseller_list = response.json()

        # getting the date info I need for the next call
        previous_published_date = bestseller_list["results"]["previous_published_date"]
        # print(previous_published_date)
        
        # Getting the dataset I need
        bestselling_books = bestseller_list["results"]["books"]
        # print(bestselling_books[0])

        # Create a dataframe with the list of books
        bestselling_books_df = pd.json_normalize(bestselling_books)

        # Adding the additional columns with dates that I need
        published_date = bestseller_list["results"]["published_date"]
        bestselling_books_df["published_date"] = published_date

        bestsellers_date = bestseller_list["results"]["bestsellers_date"]
        bestselling_books_df["bestsellers_date"] = bestsellers_date

        # Since this will run a while, I am going to print to let myself know it worked
        print("Call 1 done")

        # Adding 12 seconds of sleep so as to not exceed the NYT call limit
        sleep(12)
        
    # For the answer other than the first call, use the date previous_published_date to call the previous list
    else: 
        url = f"https://api.nytimes.com/svc/books/v3/lists/{previous_published_date}/combined-print-and-e-book-fiction.json?api-key={api_key}"
        response = requests.get(url)
        bestseller_list = response.json()

        # getting the date info I need
        previous_published_date = bestseller_list["results"]["previous_published_date"]
        # print(previous_published_date)

        # Printing the info I need to check this worked
        bestselling_books = bestseller_list["results"]["books"]
        # print(bestselling_books[0])

        # Creating a new dataframe with the new list 
        new_bestselling_books_df = pd.json_normalize(bestselling_books)

        published_date = bestseller_list["results"]["published_date"]
        new_bestselling_books_df["published_date"] = published_date

        bestsellers_date = bestseller_list["results"]["bestsellers_date"]
        new_bestselling_books_df["bestsellers_date"] = bestsellers_date

        # Appending the new dataframe to the master data frame
        bestselling_books_df = pd.concat([bestselling_books_df, new_bestselling_books_df])

        # Since this will run a while, I am going to print to let myself know it worked
        print(f"Call {i} done")

        # Adding 12 seconds of sleep so as to not exceed the NYT call limit
        sleep(12)

Call 1 done
Call 1 done
Call 2 done
Call 3 done
Call 4 done
Call 5 done
Call 6 done
Call 7 done
Call 8 done
Call 9 done
Call 10 done
Call 11 done
Call 12 done
Call 13 done
Call 14 done
Call 15 done
Call 16 done
Call 17 done
Call 18 done
Call 19 done
Call 20 done
Call 21 done
Call 22 done
Call 23 done
Call 24 done
Call 25 done
Call 26 done
Call 27 done
Call 28 done
Call 29 done
Call 30 done
Call 31 done
Call 32 done
Call 33 done
Call 34 done
Call 35 done
Call 36 done
Call 37 done
Call 38 done
Call 39 done
Call 40 done
Call 41 done
Call 42 done
Call 43 done
Call 44 done
Call 45 done
Call 46 done
Call 47 done
Call 48 done
Call 49 done
Call 50 done
Call 51 done
Call 52 done
Call 53 done
Call 54 done
Call 55 done
Call 56 done
Call 57 done
Call 58 done
Call 59 done
Call 60 done
Call 61 done
Call 62 done
Call 63 done
Call 64 done
Call 65 done
Call 66 done
Call 67 done
Call 68 done
Call 69 done
Call 70 done
Call 71 done
Call 72 done
Call 73 done
Call 74 done
Call 75 done
Call 76 done
Call 77 d

In [45]:
bestselling_books_df.shape

(9310, 30)

In [48]:
bestselling_books_df.head()

,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,book_uri,...,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links,published_date,bestsellers_date
0,,https://www.amazon.com/dp/1538728575?tag=thene...,,0,Nicholas Sparks,https://static01.nyt.com/bestsellers/images/97...,500,329,,nyt://book/b9bf792c-a853-54ce-8f33-7117f51be365,...,1,0,,THE RETURN,2026-03-06T09:36:07.62Z,1,"[{'isbn10': '', 'isbn13': '9781538728574'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
1,,https://www.amazon.com/dp/0593199308?tag=thene...,,0,Jim Butcher,https://static01.nyt.com/bestsellers/images/97...,500,331,,nyt://book/0ef41f25-7ccc-52eb-bef9-265e6e0cd6dc,...,2,0,,BATTLE GROUND,2026-03-06T09:04:31.599Z,1,"[{'isbn10': '', 'isbn13': '9780593199329'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
2,,https://www.amazon.com/dp/0525954988?tag=thene...,,0,Ken Follett,https://static01.nyt.com/bestsellers/images/97...,500,329,,nyt://book/6fd2cc42-7bff-59a1-9c5c-20514172cf5b,...,3,3,,THE EVENING AND THE MORNING,2026-03-06T08:57:01.774Z,3,"[{'isbn10': '', 'isbn13': '9780525954989'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
3,,https://www.amazon.com/dp/1982167289?tag=thene...,,0,Lana Del Rey,https://static01.nyt.com/bestsellers/images/97...,500,326,,nyt://book/d510e3d5-f83b-50b5-9c1b-c386af482ee8,...,4,0,,VIOLET BENT BACKWARDS OVER THE GRASS,2026-03-06T09:04:34.223Z,1,"[{'isbn10': '', 'isbn13': '9781982167288'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03
4,,https://www.amazon.com/dp/198481835X?tag=thene...,,0,Jodi Picoult,https://static01.nyt.com/bestsellers/images/97...,500,329,,nyt://book/24ac761d-3017-5b80-84f7-0718f5cead02,...,5,1,,THE BOOK OF TWO WAYS,2026-03-06T08:58:44.104Z,2,"[{'isbn10': '', 'isbn13': '9781984818355'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",2020-10-18,2020-10-03


In [47]:
# Saving the second batch of raw data
bestselling_books_df.to_csv("raw_data/raw_data_1.csv")

Repeating the process a 3rd time to get as much data as possible. However, only around 50 calls were made because the structure of the historic data changed. This didn't matter too much as I already have a decade of data

In [15]:
# Setting a variable to make sure the loop doesn't use all my API calls 
number_of_calls = 480

# Setting up the last date I have
current = "2012-03-11"

# Setting up an empty url variable for my calls
url = ""

# Setting up an empty variable for the date of the previous list
previous_published_date = ""

for i in range(number_of_calls):
    
    # if it is the first call, use the specific request for the current list so that we can get the dates
    if i == 0:
        url = f"https://api.nytimes.com/svc/books/v3/lists/{current}/combined-print-and-e-book-fiction.json?api-key={api_key}"
        response = requests.get(url)
        bestseller_list = response.json()

        # getting the date info I need for the next call
        previous_published_date = bestseller_list["results"]["previous_published_date"]
        # print(previous_published_date)
        
        # Getting the dataset I need
        bestselling_books = bestseller_list["results"]["books"]
        # print(bestselling_books[0])

        # Create a dataframe with the list of books
        bestselling_books_df = pd.json_normalize(bestselling_books)

        # Adding the additional columns with dates that I need
        published_date = bestseller_list["results"]["published_date"]
        bestselling_books_df["published_date"] = published_date

        bestsellers_date = bestseller_list["results"]["bestsellers_date"]
        bestselling_books_df["bestsellers_date"] = bestsellers_date

        # Since this will run a while, I am going to print to let myself know it worked
        print("Call 1 done")

        # Adding 12 seconds of sleep so as to not exceed the NYT call limit
        sleep(12)
        
    # For the answer other than the first call, use the date previous_published_date to call the previous list
    else: 
        url = f"https://api.nytimes.com/svc/books/v3/lists/{previous_published_date}/combined-print-and-e-book-fiction.json?api-key={api_key}"
        response = requests.get(url)
        bestseller_list = response.json()

        # getting the date info I need
        previous_published_date = bestseller_list["results"]["previous_published_date"]
        # print(previous_published_date)

        # Printing the info I need to check this worked
        bestselling_books = bestseller_list["results"]["books"]
        # print(bestselling_books[0])

        # Creating a new dataframe with the new list 
        new_bestselling_books_df = pd.json_normalize(bestselling_books)

        published_date = bestseller_list["results"]["published_date"]
        new_bestselling_books_df["published_date"] = published_date

        bestsellers_date = bestseller_list["results"]["bestsellers_date"]
        new_bestselling_books_df["bestsellers_date"] = bestsellers_date

        # Appending the new dataframe to the master data frame
        bestselling_books_df = pd.concat([bestselling_books_df, new_bestselling_books_df])

        # Since this will run a while, I am going to print to let myself know it worked
        print(f"Call {i} done")

        # Adding 12 seconds of sleep so as to not exceed the NYT call limit
        sleep(12)

Call 1 done
Call 1 done
Call 2 done
Call 3 done
Call 4 done
Call 5 done
Call 6 done
Call 7 done
Call 8 done
Call 9 done
Call 10 done
Call 11 done
Call 12 done
Call 13 done
Call 14 done
Call 15 done
Call 16 done
Call 17 done
Call 18 done
Call 19 done
Call 20 done
Call 21 done
Call 22 done
Call 23 done
Call 24 done
Call 25 done
Call 26 done
Call 27 done
Call 28 done
Call 29 done
Call 30 done
Call 31 done
Call 32 done
Call 33 done
Call 34 done
Call 35 done
Call 36 done
Call 37 done
Call 38 done
Call 39 done
Call 40 done
Call 41 done
Call 42 done
Call 43 done
Call 44 done
Call 45 done
Call 46 done
Call 47 done
Call 48 done
Call 49 done
Call 50 done
Call 51 done
Call 52 done
Call 53 done
Call 54 done


KeyError: 'previous_published_date'

In [17]:
bestselling_books_df.head()

,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,book_uri,...,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links,published_date,bestsellers_date
0,,http://www.amazon.com/Celebrity-Death-J-D-Robb...,,0,J. D. Robb,https://static01.nyt.com/bestsellers/images/97...,495,307,,nyt://book/d1ec5779-c211-581e-a118-572a3b136be8,...,1,0,,CELEBRITY IN DEATH,2026-03-07T17:21:15.669Z,1,"[{'isbn10': '', 'isbn13': '9781101560365'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",2012-03-11,2012-02-25
1,,http://www.amazon.com/Girl-Kicked-Hornets-Mill...,,0,Stieg Larsson,https://static01.nyt.com/bestsellers/images/97...,485,330,https://www.nytimes.com/2010/05/21/books/21boo...,nyt://book/69a936f5-8e65-5843-a4bd-bd5de4e042a7,...,2,10,https://www.nytimes.com/2010/05/30/books/revie...,THE GIRL WHO KICKED THE HORNET’S NEST,2026-03-07T17:21:17.058Z,27,"[{'isbn10': '', 'isbn13': '9780307454560'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",2012-03-11,2012-02-25
2,,http://www.amazon.com/Perfect-Blood-Hollows-Ki...,,0,Kim Harrison,https://storage.googleapis.com/du-prd/books/im...,193,128,,nyt://book/67827cb7-ad3d-5a80-8c6d-cf825d575d71,...,3,0,,A PERFECT BLOOD,2026-03-07T17:25:37.867Z,1,"[{'isbn10': '', 'isbn13': '9780062101020'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",2012-03-11,2012-02-25
3,,http://www.amazon.com/Kill-Shot-American-Assas...,,0,Vince Flynn,https://static01.nyt.com/bestsellers/images/97...,495,323,,nyt://book/4bcb310d-405c-5501-9c98-13ecd4ef8e0e,...,4,2,,KILL SHOT,2026-03-07T17:15:10.306Z,3,"[{'isbn10': '', 'isbn13': '9781439100523'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",2012-03-11,2012-02-25
4,,http://www.amazon.com/Defending-Jacob-Novel-Wi...,,0,William Landay,https://static01.nyt.com/bestsellers/images/97...,495,300,https://www.nytimes.com/2012/02/13/books/defen...,nyt://book/a89e4199-b517-571b-8bfb-8429fa7831c4,...,5,4,,DEFENDING JACOB,2026-03-07T10:12:01.817Z,4,"[{'isbn10': '', 'isbn13': '9780345527592'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",2012-03-11,2012-02-25


In [18]:
bestseller_list

{'status': 'OK',
 'copyright': 'Copyright (c) 2026 The New York Times Company. All Rights Reserved.',
 'num_results': 20,
 'last_modified': '2025-05-19T21:17:07Z',
 'results': {'display_name': 'Combined Print Fiction',
  'list_name': 'Combined Print Fiction',
  'list_name_encoded': 'combined-print-and-e-book-fiction',
  'published_date': '2011-02-20',
  'next_published_date': '2011-02-27',
  'bestsellers_date': '2011-02-05',
  'normal_list_ends_at': 20,
  'updated': 'WEEKLY',
  'list_id': 705,
  'uri': 'nyt://bestsellerslist/7ad7a805-82c5-545e-bbf2-450174529d2f',
  'books': [{'age_group': '',
    'amazon_product_url': 'http://www.amazon.com/Tick-Tock-Michael-Bennett-Book-ebook/dp/B0047Y16MG?tag=thenewyorktim-20',
    'article_chapter_link': '',
    'asterisk': 0,
    'author': 'James Patterson and Michael Ledwidge',
    'book_image': '',
    'book_image_height': 0,
    'book_image_width': 0,
    'book_review_link': '',
    'book_uri': 'nyt://book/bd4cdccd-0040-537e-b567-f9fd4ebf031f',


In [19]:
bestselling_books_df.to_csv("raw_data/raw_data_2.csv")